# 🔍 01 Exploración de Datos y Modelado de Fraude — FRAUDIA

Este notebook Jupyter forma parte de la documentación oficial y reproducible del reto de la **Aseguradora del Sur**. Demuestra el pipeline inicial de exploración de datos, cálculo de variables de riesgo y entrenamiento del modelo no supervisado Isolation Forest.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import os

print("✅ Librerías cargadas correctamente")

## 1. Carga del Dataset Estructurado
Ingestamos el archivo Excel sintético para verificar su estructura y sus 5 hojas requeridas.

In [ ]:
excel_path = '../data/dataset/Evento Datasets_Sinteticos_Fraude_500_v2.xlsx'
if os.path.exists(excel_path):
    xl = pd.ExcelFile(excel_path)
    print(f"Hojas encontradas en el Excel: {xl.sheet_names}")
    df_siniestros = pd.read_excel(excel_path, sheet_name='1_Siniestros')
    print(f"Columnas del dataset de Siniestros ({df_siniestros.shape[0]} filas):\n", list(df_siniestros.columns))
else:
    print("❌ No se encontró el dataset en la ruta relativa. Verifique la estructura del proyecto.")

## 2. Detección de Anomalías No Supervisada (Isolation Forest)
Entrenamos el modelo para detectar los siniestros atípicos basados en el monto reclamado, suma asegurada y cantidad de reclamos previos.

In [ ]:
if 'df_siniestros' in locals():
    # Limpieza de montos y variables numéricas
    df_siniestros['Monto_Reclamado'] = pd.to_numeric(df_siniestros['Monto Reclamado ($)'], errors='coerce').fillna(0)
    df_siniestros['Suma_Asegurada'] = pd.to_numeric(df_siniestros['Suma Asegurada ($)'], errors='coerce').fillna(0)
    df_siniestros['Reclamos_Previos'] = pd.to_numeric(df_siniestros['N° Reclamos Previos Asegurado'], errors='coerce').fillna(0)
    
    features = df_siniestros[['Monto_Reclamado', 'Suma_Asegurada', 'Reclamos_Previos']]
    scaler = StandardScaler()
    X = scaler.fit_transform(features)
    
    model = IsolationForest(contamination=0.15, random_state=42)
    preds = model.fit_predict(X)
    scores = model.decision_function(X)
    
    df_siniestros['Es_Anomalia'] = preds == -1
    print(f"✅ Isolation Forest entrenado. Siniestros anómalos detectados: {df_siniestros['Es_Anomalia'].sum()}")